In [ ]:
#1. IMPORTACIONES Y CONFIGURACIÓN

import arcpy
import numpy as np
import os
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt

from arcpy.sa import ExtractByMask

arcpy.env.overwriteOutput = True

print("ArcGIS Pro / ArcPy")
print("Entorno configurado correctamente.")

In [ ]:
#2. RUTAS Y PARÁMETROS

BASE = r"C:\Users\57312\Documents\Proyectos Esri\Charla"

# Predicciones generadas por TensorFlow
RUTA_NPZ = os.path.join(
    BASE, "pronosticos_convlstm_interanual_2026_2027.npz"
)

# GeoTIFF originales del pronóstico
CARPETA_PRED = os.path.join(
    BASE, "Predicciones_ConvLSTM"
)

# GeoTIFF recortados al área de estudio
CARPETA_PRED_RECORTADAS = os.path.join(
    BASE, "Predicciones_ConvLSTM_Recortadas"
)

# Históricos ya recortados
CARPETA_HIST_RECORTADOS = os.path.join(
    BASE, "Historico_Recortado"
)

# Área de estudio
RECORTE = os.path.join(BASE, "Recorte1.shp")

# Geodatabase y Mosaic Dataset del pronóstico
GDB = os.path.join(BASE, "Extracción.gdb")
MD_PRONOSTICO = os.path.join(
    GDB, "Pronostico_ConvLSTM_2026_2027"
)

# CRF
CRF = os.path.join(
    BASE, "Pronostico_ConvLSTM_2026_2027.crf"
)

# Tablas de resultados
CSV_HIST = os.path.join(
    BASE, "area_historica_2015_2026.csv"
)

CSV_PRON = os.path.join(
    BASE, "area_predicha_2026_2027.csv"
)

# Parámetros espaciales del mosaico original
XMIN = -8320427.955700001
YMIN = 593608.3473999985
CELDA = 10.0
WKID = 3857

# Resolución de clasificación del pronóstico
UMBRAL = 0.5

os.makedirs(CARPETA_PRED, exist_ok=True)
os.makedirs(CARPETA_PRED_RECORTADAS, exist_ok=True)

print("Rutas configuradas.")
print("Área de estudio:", RECORTE)
print("Umbral del pronóstico:", UMBRAL)

In [ ]:
#3. CARGAR PREDICCIONES Y FECHAS

datos = np.load(RUTA_NPZ)

predicciones = datos["predicciones"]
fechas = pd.to_datetime(datos["fechas"])

print("Predicciones:", predicciones.shape)
print("Tipo:", predicciones.dtype)
print("Primera fecha:", fechas[0].date())
print("Última fecha:", fechas[-1].date())

assert predicciones.shape[0] == len(fechas)
assert predicciones.shape[0] == 12

print("OK — 12 predicciones mensuales cargadas.")

In [ ]:
#4. CREAR GEOTIFFS GEORREFERENCIADOS

sr = arcpy.SpatialReference(WKID)

for i, fecha in enumerate(fechas):
    nombre = f"Pred_ConvLSTM_{fecha:%Y%m%d}.tif"
    salida = os.path.join(CARPETA_PRED, nombre)

    raster = arcpy.NumPyArrayToRaster(
        predicciones[i].astype(np.float32),
        arcpy.Point(XMIN, YMIN),
        CELDA,
        CELDA,
        value_to_nodata=np.nan
    )

    raster.save(salida)
    arcpy.DefineProjection_management(salida, sr)

print(f"GeoTIFF creados: {len(fechas)}")
print("Referencia espacial: WGS 1984 Web Mercator Auxiliary Sphere (WKID 3857)")
print("Tamaño de celda: 10 m")

In [ ]:
#5. CREAR / ACTUALIZAR MOSAIC DATASET MULTIDIMENSIONAL

# Si el Mosaic Dataset no existe, se crea.
if not arcpy.Exists(MD_PRONOSTICO):
    arcpy.management.CreateMosaicDataset(
        GDB,
        os.path.basename(MD_PRONOSTICO),
        sr,
        num_bands=1,
        pixel_type="32_BIT_FLOAT"
    )

# Incorporar los GeoTIFF
arcpy.management.AddRastersToMosaicDataset(
    MD_PRONOSTICO,
    "Raster Dataset",
    CARPETA_PRED,
    update_cellsize_ranges="UPDATE_CELL_SIZES",
    update_boundary="UPDATE_BOUNDARY",
    update_overviews="NO_OVERVIEWS",
    duplicate_items_action="EXCLUDE_DUPLICATES"
)

print("Mosaic Dataset actualizado.")

# Actualizar información multidimensional
arcpy.md.BuildMultidimensionalInfo(
    MD_PRONOSTICO,
    variable_field="Variable",
    dimension_fields="Date"
)

print("Información multidimensional construida.")

In [ ]:
#6. CREAR CRF MULTIDIMENSIONAL

# El CRF se crea a partir del Mosaic Dataset final.
if os.path.exists(CRF):
    print("El CRF ya existe:")
    print(CRF)
else:
    arcpy.md.CopyRasterToCRF(
        MD_PRONOSTICO,
        CRF,
        "ALL"
    )
    print("CRF creado:")
    print(CRF)

In [ ]:
#7. RECORTAR LAS 12 PREDICCIONES AL ÁREA DE ESTUDIO

rasters = sorted(
    glob.glob(os.path.join(CARPETA_PRED, "Pred_ConvLSTM_*.tif"))
)

for raster_path in rasters:
    nombre = os.path.basename(raster_path)
    salida = os.path.join(
        CARPETA_PRED_RECORTADAS,
        nombre.replace(".tif", "_recortado.tif")
    )

    resultado = ExtractByMask(raster_path, RECORTE)
    resultado.save(salida)

print(f"Predicciones recortadas: {len(rasters)}")

In [ ]:
#8. ESTADÍSTICAS HISTÓRICAS

filas = []

for raster_path in sorted(
    glob.glob(os.path.join(CARPETA_HIST_RECORTADOS, "*.tif"))
):
    nombre = os.path.basename(raster_path)

    # Extrae la fecha YYYYMMDD del nombre del raster.
    m = re.search(r"(20\d{6})", nombre)
    if not m:
        continue

    fecha = pd.to_datetime(m.group(1), format="%Y%m%d")

    arr = arcpy.RasterToNumPyArray(raster_path)

    # Históricos: 1 = agua, 0 = no agua.
    pixeles_agua = int(np.sum(arr == 1))

    filas.append({
        "Fecha": fecha,
        "Raster": nombre,
        "Pixeles_agua": pixeles_agua,
        "Area_m2": pixeles_agua * CELDA * CELDA,
        "Area_ha": pixeles_agua * CELDA * CELDA / 10000
    })

df_hist = pd.DataFrame(filas).sort_values("Fecha")

df_hist.to_csv(
    CSV_HIST,
    index=False,
    encoding="utf-8-sig"
)

print(f"Registros históricos: {len(df_hist)}")
print(f"CSV: {CSV_HIST}")

In [ ]:
#9. ESTADÍSTICAS DEL PRONÓSTICO

filas = []

for raster_path in sorted(
    glob.glob(
        os.path.join(
            CARPETA_PRED_RECORTADAS,
            "Pred_ConvLSTM_*_recortado.tif"
        )
    )
):
    nombre = os.path.basename(raster_path)

    m = re.search(r"(20\d{6})", nombre)
    if not m:
        continue

    fecha = pd.to_datetime(m.group(1), format="%Y%m%d")

    arr = arcpy.RasterToNumPyArray(raster_path).astype(np.float32)

    # Conversión de probabilidad a presencia/ausencia de agua.
    pixeles_agua = int(np.sum(arr >= UMBRAL))

    filas.append({
        "Fecha": fecha,
        "Raster": nombre,
        "Pixeles_agua": pixeles_agua,
        "Area_m2": pixeles_agua * CELDA * CELDA,
        "Area_ha": pixeles_agua * CELDA * CELDA / 10000
    })

df_pron = pd.DataFrame(filas).sort_values("Fecha")

df_pron.to_csv(
    CSV_PRON,
    index=False,
    encoding="utf-8-sig"
)

print(f"Registros pronosticados: {len(df_pron)}")
print(f"Umbral utilizado: {UMBRAL}")
print(f"CSV: {CSV_PRON}")

In [ ]:
# 10. GRÁFICA FINAL HISTÓRICO + PRONÓSTICO

df_hist["Tipo"] = "Histórico"
df_pron["Tipo"] = "Pronóstico"

serie_hist = (
    df_hist[["Fecha", "Area_ha", "Tipo"]]
    .copy()
)

serie_pron = (
    df_pron[["Fecha", "Area_ha", "Tipo"]]
    .copy()
)

serie = pd.concat(
    [serie_hist, serie_pron],
    ignore_index=True
).sort_values("Fecha")

plt.figure(figsize=(14, 6))

for tipo, grupo in serie.groupby("Tipo"):
    plt.plot(
        grupo["Fecha"],
        grupo["Area_ha"],
        marker="o",
        markersize=3,
        linewidth=1,
        label=tipo
    )

plt.axvline(
    pd.Timestamp("2026-09-01"),
    linestyle="--",
    linewidth=1
)

plt.xlabel("Fecha")
plt.ylabel("Área de agua (ha)")
plt.title("Superficie de agua: histórico y pronóstico ConvLSTM")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()